In [18]:
import json
import os

import numpy as np
import pandas as pd

from Pipeline.Model.GallstoneDataSet import GallstoneDataSet
from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.ExtremeLearningMachine import sigmoid

In [2]:
gallstone_dataset = GallstoneDataSet()
gallstone_dataset.fetch_data_path_1()
features_size = gallstone_dataset.x_train_scaled.shape[1]

In [3]:
hidden_size_explore = range(55, 59)
lambda_value_explore = 1.25 ** np.arange(-2, 2)

In [4]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x=gallstone_dataset.x_train,
    y=gallstone_dataset.y_train,
    activation_function=sigmoid,
    elm_initial_state_range= range(41, 45),
    data_split_state_range= range(1, 4),
    test_size=0.2,
    k_fold=5
)

In [5]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(hidden_size_explore)

In [6]:
hidden_node_list = EvaluationELM.extract_top_results(hidden_size_result)
best_hidden_size = hidden_node_list.iloc[0]
best_hidden_size

Hidden_Nodes                       58
Activation                    sigmoid
Lambda_Value                      0.0
avg_Accuracy_Seed_Mean         0.7219
avg_Accuracy_Seed_Std          0.0759
avg_Accuracy_Seed_Min          0.5294
avg_Accuracy_Seed_Max          0.8627
avg_Precision_Seed_Mean        0.7503
avg_Precision_Seed_Std         0.0854
avg_Precision_Seed_Min         0.5238
avg_Precision_Seed_Max         0.9048
avg_Recall_Seed_Mean           0.6581
avg_Recall_Seed_Std            0.1163
avg_Recall_Seed_Min               0.4
avg_Recall_Seed_Max              0.88
avg_NPV_Seed_Mean               0.706
avg_NPV_Seed_Std               0.0751
avg_NPV_Seed_Min               0.5333
avg_NPV_Seed_Max                 0.88
avg_Specificity_Seed_Mean      0.7833
avg_Specificity_Seed_Std       0.0904
avg_Specificity_Seed_Min          0.6
avg_Specificity_Seed_Max       0.9231
avg_F1-Score_Seed_Mean          0.697
avg_F1-Score_Seed_Std          0.0932
avg_F1-Score_Seed_Min          0.4783
avg_F1-Score

In [7]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size=best_hidden_size['Hidden_Nodes'],
    lambda_range=lambda_value_explore
)

In [8]:
lambda_value_list = EvaluationELM.extract_top_results(lambda_result)
best_lambda_value = lambda_value_list.iloc[0]
best_lambda_value

Hidden_Nodes                       58
Activation                    sigmoid
Lambda_Value                      0.8
avg_Accuracy_Seed_Mean         0.7284
avg_Accuracy_Seed_Std          0.0807
avg_Accuracy_Seed_Min          0.5882
avg_Accuracy_Seed_Max          0.8824
avg_Precision_Seed_Mean        0.7591
avg_Precision_Seed_Std         0.0917
avg_Precision_Seed_Min         0.5926
avg_Precision_Seed_Max         0.9444
avg_Recall_Seed_Mean           0.6622
avg_Recall_Seed_Std            0.1241
avg_Recall_Seed_Min              0.32
avg_Recall_Seed_Max              0.92
avg_NPV_Seed_Mean              0.7116
avg_NPV_Seed_Std               0.0814
avg_NPV_Seed_Min               0.5758
avg_NPV_Seed_Max               0.9091
avg_Specificity_Seed_Mean      0.7925
avg_Specificity_Seed_Std       0.0859
avg_Specificity_Seed_Min       0.5769
avg_Specificity_Seed_Max       0.9615
avg_F1-Score_Seed_Mean         0.7025
avg_F1-Score_Seed_Std          0.1009
avg_F1-Score_Seed_Min          0.4706
avg_F1-Score

In [9]:
size_lambda_record , size_lambda_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_node_list['Hidden_Nodes'],
    lambda_range=lambda_value_explore
)

In [10]:
size_lambda_list = EvaluationELM.extract_top_results(size_lambda_result)
best_size_lambda = size_lambda_list.iloc[0]
best_size_lambda

Hidden_Nodes                       58
Activation                    sigmoid
Lambda_Value                      0.8
avg_Accuracy_Seed_Mean         0.7284
avg_Accuracy_Seed_Std          0.0807
avg_Accuracy_Seed_Min          0.5882
avg_Accuracy_Seed_Max          0.8824
avg_Precision_Seed_Mean        0.7591
avg_Precision_Seed_Std         0.0917
avg_Precision_Seed_Min         0.5926
avg_Precision_Seed_Max         0.9444
avg_Recall_Seed_Mean           0.6622
avg_Recall_Seed_Std            0.1241
avg_Recall_Seed_Min              0.32
avg_Recall_Seed_Max              0.92
avg_NPV_Seed_Mean              0.7116
avg_NPV_Seed_Std               0.0814
avg_NPV_Seed_Min               0.5758
avg_NPV_Seed_Max               0.9091
avg_Specificity_Seed_Mean      0.7925
avg_Specificity_Seed_Std       0.0859
avg_Specificity_Seed_Min       0.5769
avg_Specificity_Seed_Max       0.9615
avg_F1-Score_Seed_Mean         0.7025
avg_F1-Score_Seed_Std          0.1009
avg_F1-Score_Seed_Min          0.4706
avg_F1-Score

In [11]:
model_configs_payload = [
    {
        "Model_Types" : "Best_Hidden_Nodes",
        "Hidden_Nodes": int(best_hidden_size['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_hidden_size['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Lambda",
        "Hidden_Nodes": int(best_lambda_value['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_lambda_value['Lambda_Value'])
    },
    {
        "Model_Types" : "Best_Size_and_Lambda",
        "Hidden_Nodes": int(best_size_lambda['Hidden_Nodes']),
        "Activation": "sigmoid",
        "Lambda_Value": float(best_size_lambda['Lambda_Value'])
    }
]

In [12]:
dataset_dir = '../../Dataset/JSON/'
config_file = os.path.join(dataset_dir, 'full_model_configs.json')

with open(config_file, 'w') as f:
    json.dump(model_configs_payload, f, indent=4)

print(f"Successfully initialized {config_file} with {len(model_configs_payload)} configurations.")

Successfully initialized ../../Dataset/JSON/full_model_configs.json with 3 configurations.


In [19]:
s= pd.DataFrame(size_lambda_result)
s

,Hidden_Nodes,Activation,Lambda_Value,avg_Accuracy_Seed_Mean,avg_Accuracy_Seed_Std,avg_Accuracy_Seed_Min,avg_Accuracy_Seed_Max,avg_Precision_Seed_Mean,avg_Precision_Seed_Std,avg_Precision_Seed_Min,...,avg_F2-Score_Seed_Min,avg_F2-Score_Seed_Max,avg_Bal Accuracy_Seed_Mean,avg_Bal Accuracy_Seed_Std,avg_Bal Accuracy_Seed_Min,avg_Bal Accuracy_Seed_Max,avg_MCC_Seed_Mean,avg_MCC_Seed_Std,avg_MCC_Seed_Min,avg_MCC_Seed_Max
0,58,sigmoid,0.64,0.7278,0.0803,0.5882,0.8824,0.7591,0.0906,0.5833,...,0.3241,0.8915,0.7266,0.0810,0.5854,0.8815,0.4616,0.1599,0.1757,0.7666
1,58,sigmoid,0.80,0.7284,0.0807,0.5882,0.8824,0.7591,0.0917,0.5926,...,0.3670,0.8915,0.7274,0.0813,0.5854,0.8815,0.4624,0.1613,0.1764,0.7666
2,58,sigmoid,1.00,0.7248,0.0796,0.5882,0.8824,0.7557,0.0916,0.5926,...,0.3670,0.8594,0.7238,0.0802,0.5854,0.8815,0.4553,0.1589,0.1764,0.7666
3,58,sigmoid,1.25,0.7225,0.0766,0.5882,0.8431,0.7543,0.0919,0.5909,...,0.4091,0.8594,0.7216,0.0771,0.5854,0.8423,0.4510,0.1537,0.1755,0.7047
4,56,sigmoid,0.64,0.7235,0.0772,0.5490,0.8627,0.7593,0.0889,0.5455,...,0.4018,0.8915,0.7224,0.0777,0.5477,0.8623,0.4542,0.1547,0.0963,0.7257
5,56,sigmoid,0.80,0.7212,0.0774,0.5490,0.8627,0.7573,0.0915,0.5455,...,0.4018,0.8915,0.7201,0.0779,0.5477,0.8623,0.4499,0.1556,0.0963,0.7257
6,56,sigmoid,1.00,0.7199,0.0768,0.5490,0.8431,0.7528,0.0908,0.5455,...,0.3982,0.8915,0.7188,0.0772,0.5477,0.8446,0.4466,0.1545,0.0963,0.6957
7,56,sigmoid,1.25,0.7199,0.0742,0.5686,0.8627,0.7516,0.0889,0.5556,...,0.3982,0.8915,0.7189,0.0745,0.5669,0.8615,0.4463,0.1492,0.1360,0.7298
8,55,sigmoid,0.64,0.7088,0.0811,0.5490,0.8431,0.7432,0.0925,0.5455,...,0.4237,0.8333,0.7075,0.0816,0.5462,0.8423,0.4230,0.1629,0.0963,0.6878
9,55,sigmoid,0.80,0.7095,0.0800,0.5490,0.8431,0.7424,0.0924,0.5455,...,0.4622,0.8333,0.7083,0.0805,0.5477,0.8423,0.4241,0.1609,0.0963,0.6938
